In [32]:
!pip install pandas numpy requests pandas_datareader gdeltdoc

In [34]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime

print("Libraries loaded successfully")

Libraries loaded successfully


In [60]:
from pandas_datareader import data as pdr
import datetime

# Période d'étude
start = datetime.datetime(2018,4,1)
end = datetime.datetime(2020,6,30)

# Bund 10Y (Allemagne)
bund = pdr.DataReader("IRLTLT01DEM156N", "fred", start, end)
# ticker : IRLTLT01DEM156N : serie mensuelle de fred : data base de la FED

# France 10Y
france = pdr.DataReader("IRLTLT01FRM156N", "fred", start, end)

# Construction du spread
spread = france.iloc[:,0] - bund.iloc[:,0]
spread = spread.to_frame(name="spread")

spread.head()

,spread
DATE,
2018-04-01,0.300000
2018-05-01,0.340000
2018-06-01,0.422857
2018-07-01,0.395455
2018-08-01,0.409130


In [48]:
# Variation mensuelle (regression : delta)
spread["delta"] = spread["spread"].diff()

spread.head() #head affiche que les 5 premieres lignes 

,spread,delta
DATE,,
2024-01-01,0.565455,NaN
2024-02-01,0.518095,-0.047359
2024-03-01,0.470500,-0.047595
2024-04-01,0.524286,0.053786
2024-05-01,0.510455,-0.013831


In [49]:
# Direction du spread : 1 : spread monte // 0 : baisse (classification)
spread["direction"] = (spread["delta"] > 0).astype(int)

spread.head()

,spread,delta,direction
DATE,,,
2024-01-01,0.565455,NaN,0
2024-02-01,0.518095,-0.047359,0
2024-03-01,0.470500,-0.047595,0
2024-04-01,0.524286,0.053786,1
2024-05-01,0.510455,-0.013831,0


In [ ]:
# Volatilité glissante sur 6 mois du spread (rolling std)
spread["vol_6m"] = spread["delta"].rolling(6).std()
spread.head()

,spread,delta,direction,vol_6m
DATE,,,,
2024-01-01,0.565455,NaN,0,NaN
2024-02-01,0.518095,-0.047359,0,NaN
2024-03-01,0.470500,-0.047595,0,NaN
2024-04-01,0.524286,0.053786,1,NaN
2024-05-01,0.510455,-0.013831,0,NaN


In [ ]:
# ca marche pas mais je le laisse au cas ou lol

import requests
import pandas as pd
import time

API_KEY = "bc67b7baf6e296165d1df5a60a69acab"

all_articles = []

for year in range(2024, 2026):
    for month in range(1, 13):
        
        start_date = f"{year}-{month:02d}-01"
        
        # calcul date fin mois simple
        if month == 12:
            end_date = f"{year}-12-31"
        else:
            end_date = f"{year}-{month+1:02d}-01"
        
        print(f"Downloading {start_date}...")
        
        url = "https://gnews.io/api/v4/search"
        
        params = {
            "q": "Europe OR eurozone OR France OR Germany",
            "from": start_date,
            "to": end_date,
            "lang": "en",
            "max": 100,
            "token": API_KEY
        }
        
        try:
            response = requests.get(url, params=params)
            data = response.json()
            
            if "articles" in data:
                articles = data["articles"]
                if len(articles) > 0:
                    all_articles.extend(articles)
            
            time.sleep(1)  # évite blocage API
            
        except Exception as e:
            print(f"Error {start_date}: {e}")

# Création dataframe final
news_df = pd.DataFrame(all_articles)

print("Total articles:", len(news_df))
news_df.head()

Total articles: 240


,id,title,description,content,url,image,publishedAt,lang,source
0,ed2bd8ca41eb28b1a9f65f70ac7ce0ab,France pushes for return of intellectually dis...,Intellectually impaired athletes have been sid...,"LANS-EN-VERCORS, France — On a well-groomed, s...",https://www.adn.com/sports/national-sports/202...,https://www.adn.com/resizer/v2/WOC5BMXFFCAMDL5...,2026-03-03T02:35:59Z,en,"{'id': '3ae5ebb8ffc400c9f5ec57c1814b41fa', 'na..."
1,35d76232a754f4884921399a541a5154,Europe defends military bases and struggles to...,"Britain, France, and Germany say they work wit...",BRUSSELS — The U.S.-Israeli war on Iran and Te...,https://www.adn.com/nation-world/2026/03/02/eu...,https://www.adn.com/resizer/v2/CO5JPYT24HPDSZE...,2026-03-03T02:28:22Z,en,"{'id': '3ae5ebb8ffc400c9f5ec57c1814b41fa', 'na..."
2,3fa6d7c4f1860ea2698344ca0233b78a,"France to boost nuclear arsenal, involve Europ...",PARIS (Reuters) -- France will expand its nucl...,PARIS (Reuters) -- France will expand its nucl...,https://asia.nikkei.com/politics/defense/franc...,https://images.ft.com/v3/image/raw/https%3A%2F...,2026-03-03T02:23:17Z,en,"{'id': '8b41f671a9a237f65861978dca3609d0', 'na..."
3,e7be21bc8f82565f174b2a3067d3ab8f,Germany scrambles to rescue thousands of citiz...,A plane left Abu Dhabi for Munich with no pass...,Follow our live coverage here.\nBERLIN – Germa...,https://www.straitstimes.com/world/europe/germ...,https://cassette.sphdigital.com.sg/image/strai...,2026-03-03T02:18:00Z,en,"{'id': '72bbcd9c095a3d51c97cdda7a97dba31', 'na..."
4,69ccc1f226c57cd62d52ea080b6a6314,Strait of Hormuz impasse squeezes world shipping,"PARIS, France — With few captains willing to b...","PARIS, France — With few captains willing to b...",https://www.manilatimes.net/2026/03/03/world/a...,https://cdnx.premiumread.com/?url=https://www....,2026-03-03T02:10:00Z,en,"{'id': '7c666ee953f873d3057b310220f510df', 'na..."


In [54]:
print(len(articles))


10


In [55]:
news_df.columns

# Construire texte complet
news_df["text"] = (
    news_df["title"].fillna("") + " " +
    news_df["description"].fillna("") + " " +
    news_df["content"].fillna("")
)

news_df["text"] = news_df["text"].str.lower()

# Transformer la date
news_df["publishedAt"] = pd.to_datetime(news_df["publishedAt"])
news_df["month"] = news_df["publishedAt"].dt.to_period("M")

# Definir les mots clés
inflation_words = [
    "inflation", "price pressure", "cpi", "hike", "rate increase",
    "monetary policy", "hawkish", "dovish", "tightening", "ecb"
]

risk_words = [
    "crisis", "recession", "debt", "deficit", "default",
    "downgrade", "risk premium", "volatility", "instability"
]

growth_words = [
    "recovery", "expansion", "gdp", "growth",
    "unemployment", "employment"
]

geopolitical_words = [
    "war", "conflict", "energy shock", "gas", "sanctions"
]

# Fonction de comptage 
def count_keywords(text, keywords):
    count = 0
    for word in keywords:
        count += text.count(word)
    return count

# Creer les indices 
news_df["macro_inflation_index"] = news_df["text"].apply(
    lambda x: count_keywords(x, inflation_words)
)

news_df["macro_risk_index"] = news_df["text"].apply(
    lambda x: count_keywords(x, risk_words)
)

news_df["growth_index"] = news_df["text"].apply(
    lambda x: count_keywords(x, growth_words)
)

news_df["geopolitical_index"] = news_df["text"].apply(
    lambda x: count_keywords(x, geopolitical_words)
)

# Agregation
monthly_indices = news_df.groupby("month")[[
    "macro_inflation_index",
    "macro_risk_index",
    "growth_index",
    "geopolitical_index"
]].sum()

monthly_indices.head()


/Library/Frameworks/Python.framework/Versions/3.8/lib/python3.8/site-packages/pandas/core/arrays/datetimes.py:1162: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  warnings.warn(


,macro_inflation_index,macro_risk_index,growth_index,geopolitical_index
month,,,,
2026-03,0,0,0,240


In [56]:
# Fusion news avec spread 

spread["month"] = spread.index.to_period("M")

data_final = spread.merge(monthly_indices, on="month", how="left")

data_final.fillna(0, inplace=True)

data_final.head()

,spread,delta,direction,vol_6m,month,macro_inflation_index,macro_risk_index,growth_index,geopolitical_index
0,0.565455,0.000000,0,0.0,2024-01,0.0,0.0,0.0,0.0
1,0.518095,-0.047359,0,0.0,2024-02,0.0,0.0,0.0,0.0
2,0.470500,-0.047595,0,0.0,2024-03,0.0,0.0,0.0,0.0
3,0.524286,0.053786,1,0.0,2024-04,0.0,0.0,0.0,0.0
4,0.510455,-0.013831,0,0.0,2024-05,0.0,0.0,0.0,0.0


In [61]:
import pandas as pd

news_df = pd.read_csv("../data/reuters_headlines.csv")

news_df.head()

news_df.columns

Index(['Headlines', 'Time', 'Description'], dtype='object')

In [62]:
# Construire texte complet
news_df["text"] = (
    news_df["Headlines"].fillna("") + " " +
    news_df["Description"].fillna("")
)

news_df["text"] = news_df["text"].str.lower()


# Transformer la date (att° regarde la période exacte (probablement 2017–2020)
news_df["Time"] = pd.to_datetime(news_df["Time"])
news_df["month"] = news_df["Time"].dt.to_period("M")

news_df["month"].min(), news_df["month"].max()

# Definir les mots-clés macro
inflation_words = [
    "inflation", "price pressure", "cpi", "hike",
    "rate increase", "monetary policy", "hawkish",
    "dovish", "tightening", "ecb"
]

risk_words = [
    "crisis", "recession", "debt", "deficit",
    "default", "downgrade", "risk premium",
    "volatility", "instability"
]

growth_words = [
    "recovery", "expansion", "gdp", "growth",
    "unemployment", "employment"
]

geopolitical_words = [
    "war", "conflict", "energy shock",
    "gas", "sanctions"
]

# Fonction de comptage 
def count_keywords(text, keywords):
    return sum(text.count(word) for word in keywords)

# Creer les indices : 
news_df["macro_inflation_index"] = news_df["text"].apply(
    lambda x: count_keywords(x, inflation_words)
)

news_df["macro_risk_index"] = news_df["text"].apply(
    lambda x: count_keywords(x, risk_words)
)

news_df["growth_index"] = news_df["text"].apply(
    lambda x: count_keywords(x, growth_words)
)

news_df["geopolitical_index"] = news_df["text"].apply(
    lambda x: count_keywords(x, geopolitical_words)
)

# Agregation mensuelle 
monthly_indices = news_df.groupby("month")[[
    "macro_inflation_index",
    "macro_risk_index",
    "growth_index",
    "geopolitical_index"
]].sum()

monthly_indices.head()

,macro_inflation_index,macro_risk_index,growth_index,geopolitical_index
month,,,,
2018-03,26,25,23,110
2018-04,66,35,70,231
2018-05,39,46,61,205
2018-06,70,41,82,257
2018-07,49,35,103,245


In [ ]:
## Preparer le spread mensuel

# 1) convertir index en datetime si besoin
spread.index = pd.to_datetime(spread.index)

# 2️) passer en mensuel (moyenne du mois)
spread_monthly = spread.resample("M").mean()

# 3️) créer variable month compatible avec news
spread_monthly["month"] = spread_monthly.index.to_period("M")

spread_monthly.head()

In [ ]:
# Merge news + spread
data_final = spread_monthly.merge(
    monthly_indices,
    on="month",
    how="left"
)

data_final.fillna(0, inplace=True)

data_final.head()

In [ ]:
# PROJET ML
# Creer la target 

data_final["delta"] = data_final["spread"].diff()

data_final["direction"] = (data_final["delta"] > 0).astype(int)

data_final.head()


# Preparer X et y
features = [
    "macro_inflation_index",
    "macro_risk_index",
    "growth_index",
    "geopolitical_index"
]

X = data_final[features]
y = data_final["direction"]


# Modele ML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle=False
)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

#Importance economique 
import pandas as pd

importance = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

importance